# Plan-and-Execute — Planning Before Acting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/06_plan_and_execute.ipynb)

## What This Notebook Teaches You

ReAct agents are **reactive**: they decide one step at a time. Plan-and-Execute agents are **proactive**: they create a full plan before executing any step. This is the difference between "improvising a speech" and "writing an outline first."

**Research foundation**: Wang et al. 2023, *"Plan-and-Solve Prompting: Improving Zero-Shot Chain-of-Thought Reasoning by Large Language Models"* showed that asking LLMs to plan before solving boosts performance on multi-step reasoning tasks.

By the end of this notebook, you will:

1. **Understand why planning helps** — when reactive agents fail and proactive planning wins
2. **Build a PlannerAgent** that decomposes tasks into dependency-aware step plans (JSON)
3. **Build an ExecutorAgent** that runs each step with ReAct-style tool use
4. **Combine them** in a full PlanAndExecuteAgent coordinator
5. **Implement adaptive re-planning** — when a step fails, re-plan from current progress
6. **Compare Plan-and-Execute vs ReAct** head-to-head on complex tasks

---

> **Prerequisites**: Notebooks 02 (tool use) and 03 (ReAct).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~50–65 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **Why Plan First?**
- Understand **Architecture Overview**
- Understand **Building the Components**


## Part 0: Environment Setup

We load **Qwen3-8B** with 4-bit quantization. The same model plays both Planner and Executor roles — the behavior changes based on the system prompt.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Plan First?

### 1.1 — The Problem with Reactive Agents

ReAct agents decide **one action at a time**. This works well for simple tasks, but fails for complex ones:

| Scenario | ReAct Behavior | Problem |
|----------|---------------|---------|
| Multi-step research | Searches for each fact independently | Misses dependencies between facts |
| Travel planning | Books hotel before checking flight availability | Wasted steps, backtracking |
| Data analysis | Runs queries without a strategy | Redundant computations, missed insights |

### 1.2 — When Planning Helps

Planning is valuable when:
- **Tasks have dependencies** — step 3 needs results from steps 1 and 2
- **Errors are expensive** — wrong actions waste tokens/time and may be irreversible
- **The goal is complex** — more than 3-4 steps required
- **Order matters** — some sequences are much more efficient than others

### 1.3 — ReAct vs Plan-and-Execute

```
ReAct (Reactive):                    Plan-and-Execute (Proactive):

Task → Think → Act → Observe →      Task → PLAN (all steps) → Execute Step 1
       Think → Act → Observe →                                → Execute Step 2
       Think → Act → Observe →                                → Execute Step 3
       ... (no global view)                                   → Combine Results

✗ Local decisions only               ✓ Global strategy first
✗ Can go in circles                  ✓ Clear progression
✗ No resource budgeting              ✓ Known step count
✓ Flexible, adapts easily            △ Re-planning needed on failure
```

## 2. Architecture Overview

### 2.1 — The Plan-and-Execute Pattern

```
┌─────────────────────────────────────────────────────┐
│                 PlanAndExecuteAgent                  │
│                                                     │
│  ┌──────────────┐      ┌──────────────────────────┐ │
│  │ PlannerAgent  │      │     ExecutorAgent         │ │
│  │              │      │                          │ │
│  │ Takes task   │      │ Takes single step +      │ │
│  │ Returns JSON │──────│ context from prior steps  │ │
│  │ plan with    │      │                          │ │
│  │ dependencies │      │ Uses tools (calculator,   │ │
│  │              │      │ knowledge_search) in a    │ │
│  └──────────────┘      │ ReAct loop               │ │
│         │              └──────────────────────────┘ │
│         │                        │                  │
│         ▼                        ▼                  │
│  ┌──────────┐           ┌──────────────┐            │
│  │   Plan   │           │ Step Results │            │
│  │ (steps + │           │ (accumulated │            │
│  │  deps)   │           │  context)    │            │
│  └──────────┘           └──────────────┘            │
│                                                     │
│  On failure: Re-plan with progress + error info     │
└─────────────────────────────────────────────────────┘
```

The key insight: **separation of concerns**. The Planner thinks strategically (what steps, in what order). The Executor thinks tactically (how to accomplish each step using tools).

## 3. Building the Components

### 3.1 — Data Structures

We define clean data structures for plans and step results.

In [ ]:
@dataclass
class PlanStep:
    """A single step in a plan."""
    step: int
    description: str
    depends_on: List[int] = field(default_factory=list)
    status: str = "pending"  # pending, running, completed, failed
    result: Optional[str] = None

@dataclass
class Plan:
    """A complete plan with ordered steps."""
    task: str
    steps: List[PlanStep] = field(default_factory=list)
    
    def get_ready_steps(self) -> List[PlanStep]:
        """Return steps whose dependencies are all completed."""
        completed = {s.step for s in self.steps if s.status == "completed"}
        return [
            s for s in self.steps
            if s.status == "pending" and all(d in completed for d in s.depends_on)
        ]
    
    def get_context_for_step(self, step: PlanStep) -> str:
        """Gather results from dependency steps as context."""
        context_parts = []
        for dep_num in step.depends_on:
            dep_step = next(s for s in self.steps if s.step == dep_num)
            if dep_step.result:
                context_parts.append(f"[Step {dep_num} result]: {dep_step.result}")
        return "\n".join(context_parts) if context_parts else "No prior context."
    
    def summary(self) -> str:
        lines = [f"Plan for: {self.task}"]
        for s in self.steps:
            deps = f" (depends on: {s.depends_on})" if s.depends_on else ""
            lines.append(f"  Step {s.step}: [{s.status}] {s.description}{deps}")
            if s.result:
                lines.append(f"    → {s.result[:120]}...")
        return "\n".join(lines)

# Test the data structures
plan = Plan(task="example", steps=[
    PlanStep(step=1, description="First step"),
    PlanStep(step=2, description="Second step", depends_on=[1]),
    PlanStep(step=3, description="Third step", depends_on=[1, 2]),
])
print("Ready steps:", [s.step for s in plan.get_ready_steps()])
plan.steps[0].status = "completed"
plan.steps[0].result = "Step 1 done."
print("After step 1 completes:", [s.step for s in plan.get_ready_steps()])
print("Context for step 2:", plan.get_context_for_step(plan.steps[1]))

### 3.2 — The Planner Agent

The PlannerAgent takes a task description and produces a structured JSON plan. The key is a system prompt that enforces the output format.

In [ ]:
PLANNER_SYSTEM_PROMPT = """You are a planning agent. Given a task, create a step-by-step plan.

RULES:
1. Break the task into 3-7 concrete, actionable steps
2. Each step should be self-contained enough for another agent to execute
3. Identify dependencies between steps
4. Output ONLY valid JSON in this exact format:

{"steps": [
  {"step": 1, "description": "...", "depends_on": []},
  {"step": 2, "description": "...", "depends_on": [1]},
  {"step": 3, "description": "...", "depends_on": [1, 2]}
]}

Do NOT include any text before or after the JSON."""

def parse_plan(raw_output: str, task: str) -> Plan:
    """Parse LLM output into a Plan object."""
    # Extract JSON from the response
    json_match = re.search(r'\{[\s\S]*\}', raw_output)
    if not json_match:
        raise ValueError(f"No JSON found in planner output: {raw_output[:200]}")
    
    plan_data = json.loads(json_match.group())
    steps = []
    for s in plan_data["steps"]:
        steps.append(PlanStep(
            step=s["step"],
            description=s["description"],
            depends_on=s.get("depends_on", [])
        ))
    return Plan(task=task, steps=steps)

class PlannerAgent:
    """Agent that creates structured plans for complex tasks."""
    
    def __init__(self):
        self.system_prompt = PLANNER_SYSTEM_PROMPT
    
    def create_plan(self, task: str) -> Plan:
        """Generate a plan for the given task."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Create a plan for: {task}"}
        ]
        raw = generate(messages, max_new_tokens=600, temperature=0.3, do_sample=True)
        print(f"[Planner] Raw output:\n{raw}\n")
        return parse_plan(raw, task)
    
    def replan(self, task: str, completed_steps: List[dict], failure_info: str) -> Plan:
        """Re-plan after a failure, incorporating progress so far."""
        context = "COMPLETED STEPS:\n"
        for s in completed_steps:
            context += f"  Step {s['step']}: {s['description']} → {s['result']}\n"
        context += f"\nFAILURE: {failure_info}\n"
        context += "\nCreate a NEW plan for the REMAINING work. Reuse completed results."
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Task: {task}\n\n{context}"}
        ]
        raw = generate(messages, max_new_tokens=600, temperature=0.3, do_sample=True)
        print(f"[Planner] Re-plan output:\n{raw}\n")
        return parse_plan(raw, task)

# Test the planner
planner = PlannerAgent()
test_plan = planner.create_plan(
    "Calculate the total cost of a road trip from NYC to LA, considering gas prices, "
    "distance, hotel stays, and food budget."
)
print("\n" + test_plan.summary())

### 3.3 — Tools for the Executor

The Executor needs tools to accomplish steps. We provide a calculator and a knowledge search.

In [ ]:
# Tool definitions for the executor
TOOLS = {
    "calculator": {
        "description": "Evaluate a mathematical expression. Input: a math expression string.",
        "function": lambda expr: str(eval(expr, {"__builtins__": {}}, {"math": math}))
    },
    "knowledge_search": {
        "description": "Search a knowledge base for facts. Input: a search query string.",
        "function": lambda query: _knowledge_search(query)
    }
}

# Simple knowledge base for demonstrations
KNOWLEDGE_BASE = {
    "nyc to la distance": "The driving distance from New York City to Los Angeles is approximately 2,790 miles via I-80 W and I-76 W.",
    "average gas price us": "The average gas price in the US is approximately $3.50 per gallon (2024).",
    "average car mpg": "The average fuel economy for passenger cars in the US is about 25 miles per gallon.",
    "average hotel price us": "The average hotel price along major US highways is $100-150 per night.",
    "daily food budget travel": "A moderate daily food budget for road trip travel is $40-60 per person.",
    "speed of light": "The speed of light in vacuum is 299,792,458 meters per second.",
    "earth population": "The estimated world population is approximately 8.1 billion people (2024).",
    "python prime sieve": "The Sieve of Eratosthenes finds all primes up to n by iteratively marking multiples.",
    "photosynthesis": "Photosynthesis converts CO2 and water into glucose and oxygen using sunlight. The equation: 6CO2 + 6H2O + light → C6H12O6 + 6O2.",
    "us gdp 2023": "US GDP in 2023 was approximately $27.36 trillion.",
    "renewable energy share": "Renewable energy accounts for about 30% of global electricity generation (2023).",
    "mars distance from earth": "Mars is between 34 million and 250 million miles from Earth depending on orbital positions.",
}

def _knowledge_search(query: str) -> str:
    """Search the knowledge base for relevant facts."""
    query_lower = query.lower()
    results = []
    for key, value in KNOWLEDGE_BASE.items():
        if any(word in key for word in query_lower.split()):
            results.append(value)
    if results:
        return " | ".join(results[:3])
    return f"No specific results found for '{query}'. Use your best knowledge to estimate."

# Format tool descriptions for the prompt
TOOL_DESC = "\n".join(
    f"- {name}: {info['description']}" for name, info in TOOLS.items()
)
print("Available tools:")
print(TOOL_DESC)
print("\nTest knowledge_search:", _knowledge_search("gas price"))
print("Test calculator:", TOOLS["calculator"]["function"]("2790 / 25 * 3.50"))

### 3.4 — The Executor Agent

The ExecutorAgent is a focused ReAct-style agent that executes **one step** of the plan. It receives the step description plus context from completed dependency steps.

In [ ]:
EXECUTOR_SYSTEM_PROMPT = f"""You are an executor agent. You will be given a single task step to complete.

You have access to these tools:
{TOOL_DESC}

To use a tool, write EXACTLY this format:
Action: tool_name
Input: tool_input

Then wait for the observation. When you have the final answer, write:
Final Answer: your answer here

RULES:
- Focus ONLY on the step you are given
- Use the context from prior steps if provided
- Use tools when you need factual data or calculations
- Provide a clear, specific answer"""

class ExecutorAgent:
    """ReAct-style agent that executes a single plan step."""
    
    def __init__(self, max_iterations: int = 5):
        self.max_iterations = max_iterations
    
    def execute(self, step_description: str, context: str = "") -> str:
        """Execute a single step and return the result."""
        user_msg = f"Step to execute: {step_description}"
        if context and context != "No prior context.":
            user_msg += f"\n\nContext from prior steps:\n{context}"
        
        messages = [
            {"role": "system", "content": EXECUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ]
        
        for iteration in range(self.max_iterations):
            response = generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)
            print(f"  [Executor iter {iteration+1}] {response[:200]}...")
            
            # Check for final answer
            if "Final Answer:" in response:
                answer = response.split("Final Answer:")[-1].strip()
                return answer
            
            # Check for tool use
            action_match = re.search(r'Action:\s*(.+)', response)
            input_match = re.search(r'Input:\s*(.+)', response)
            
            if action_match and input_match:
                tool_name = action_match.group(1).strip()
                tool_input = input_match.group(1).strip()
                
                if tool_name in TOOLS:
                    try:
                        observation = TOOLS[tool_name]["function"](tool_input)
                    except Exception as e:
                        observation = f"Error: {str(e)}"
                else:
                    observation = f"Unknown tool: {tool_name}. Available: {list(TOOLS.keys())}"
                
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {observation}"})
                print(f"  [Tool] {tool_name}({tool_input}) → {observation[:100]}")
            else:
                # No tool call and no final answer — treat as final answer
                return response.strip()
        
        return "Step execution reached max iterations without a final answer."

# Test the executor
executor = ExecutorAgent()
result = executor.execute(
    "Find the driving distance from NYC to LA",
    context=""
)
print(f"\nExecutor result: {result}")

## 4. The Plan-and-Execute Coordinator

Now we combine the Planner and Executor into a single coordinating agent.

In [ ]:
class PlanAndExecuteAgent:
    """Coordinator that plans first, then executes each step."""
    
    def __init__(self, max_replans: int = 2):
        self.planner = PlannerAgent()
        self.executor = ExecutorAgent()
        self.max_replans = max_replans
        self.execution_log = []
    
    def run(self, task: str, verbose: bool = True) -> str:
        """Plan and execute a complex task."""
        if verbose:
            print(f"{'='*60}")
            print(f"TASK: {task}")
            print(f"{'='*60}\n")
        
        # Phase 1: Create the plan
        if verbose:
            print("[Phase 1: PLANNING]")
        plan = self.planner.create_plan(task)
        if verbose:
            print(plan.summary())
            print()
        
        replan_count = 0
        
        # Phase 2: Execute steps
        while True:
            ready_steps = plan.get_ready_steps()
            
            if not ready_steps:
                # Check if all done or stuck
                pending = [s for s in plan.steps if s.status == "pending"]
                if not pending:
                    break  # All steps completed
                
                # Stuck — all pending steps have unmet dependencies
                failed = [s for s in plan.steps if s.status == "failed"]
                if failed and replan_count < self.max_replans:
                    if verbose:
                        print(f"\n[RE-PLANNING] (attempt {replan_count + 1})")
                    completed_info = [
                        {"step": s.step, "description": s.description, "result": s.result}
                        for s in plan.steps if s.status == "completed"
                    ]
                    failure_info = "; ".join(
                        f"Step {s.step} failed: {s.result}" for s in failed
                    )
                    plan = self.planner.replan(task, completed_info, failure_info)
                    replan_count += 1
                    if verbose:
                        print(plan.summary())
                else:
                    break
            
            for step in ready_steps:
                step.status = "running"
                if verbose:
                    print(f"\n[EXECUTING Step {step.step}]: {step.description}")
                
                context = plan.get_context_for_step(step)
                try:
                    result = self.executor.execute(step.description, context)
                    step.status = "completed"
                    step.result = result
                    self.execution_log.append({
                        "step": step.step,
                        "description": step.description,
                        "result": result,
                        "status": "completed"
                    })
                    if verbose:
                        print(f"  ✓ Result: {result[:200]}")
                except Exception as e:
                    step.status = "failed"
                    step.result = str(e)
                    self.execution_log.append({
                        "step": step.step,
                        "description": step.description,
                        "result": str(e),
                        "status": "failed"
                    })
                    if verbose:
                        print(f"  ✗ Failed: {e}")
        
        # Phase 3: Combine results
        if verbose:
            print(f"\n{'='*60}")
            print("[FINAL SUMMARY]")
            print(plan.summary())
            print(f"{'='*60}")
        
        final_results = "\n".join(
            f"Step {s.step} ({s.description}): {s.result}"
            for s in plan.steps if s.status == "completed"
        )
        return final_results
    
    def get_stats(self) -> dict:
        """Return execution statistics."""
        return {
            "total_steps": len(self.execution_log),
            "completed": sum(1 for s in self.execution_log if s["status"] == "completed"),
            "failed": sum(1 for s in self.execution_log if s["status"] == "failed"),
        }

print("✓ PlanAndExecuteAgent defined")

## 5. Running Complex Tasks

### 5.1 — Task 1: Road Trip Cost Calculation

A multi-step task requiring research, calculation, and synthesis.

In [ ]:
agent = PlanAndExecuteAgent()
result1 = agent.run(
    "Calculate the total cost of a road trip from New York City to Los Angeles for 2 people, "
    "including gas, hotels (4 nights), and food."
)
print("\nStats:", agent.get_stats())

### 5.2 — Task 2: Scientific Comparison

A research-heavy task where order matters — you need facts before you can compare.

In [ ]:
agent2 = PlanAndExecuteAgent()
result2 = agent2.run(
    "Compare the energy output of the Sun to total global electricity consumption. "
    "Express the ratio and explain what it means for solar energy potential."
)
print("\nStats:", agent2.get_stats())

### 5.3 — Task 3: Multi-Factor Decision Analysis

A task requiring structured analysis with dependencies.

In [ ]:
agent3 = PlanAndExecuteAgent()
result3 = agent3.run(
    "Analyze whether it's more cost-effective to buy or lease a $35,000 car over 5 years, "
    "assuming 5% interest rate, $3,000 down payment, and 12,000 miles/year."
)
print("\nStats:", agent3.get_stats())

## 6. Adaptive Re-Planning

Real-world tasks don't always go according to plan. What happens when a step fails? The Plan-and-Execute pattern handles this by **re-planning**: asking the Planner to create a new plan that accounts for what has been completed and what went wrong.

### 6.1 — Simulating Failure and Recovery

In [ ]:
class FailableExecutorAgent(ExecutorAgent):
    """Executor that can simulate failures for testing re-planning."""
    
    def __init__(self, fail_steps: List[int] = None, **kwargs):
        super().__init__(**kwargs)
        self.fail_steps = fail_steps or []
        self.current_step = 0
    
    def execute(self, step_description: str, context: str = "") -> str:
        self.current_step += 1
        if self.current_step in self.fail_steps:
            raise RuntimeError(
                f"Simulated failure on step {self.current_step}: "
                f"Tool 'knowledge_search' returned no results for this query."
            )
        return super().execute(step_description, context)

class FailablePlanAndExecuteAgent(PlanAndExecuteAgent):
    """P&E agent with a failable executor for testing re-planning."""
    
    def __init__(self, fail_steps: List[int] = None, **kwargs):
        super().__init__(**kwargs)
        self.executor = FailableExecutorAgent(fail_steps=fail_steps)

# Test re-planning: fail on step 2, observe recovery
replan_agent = FailablePlanAndExecuteAgent(fail_steps=[2], max_replans=2)
result_replan = replan_agent.run(
    "Find the population of the 3 largest US cities and calculate their total."
)
print("\nRe-planning stats:", replan_agent.get_stats())

## 7. Head-to-Head: Plan-and-Execute vs ReAct

Let's compare the two approaches on the same tasks. We'll implement a minimal ReAct agent and run both on identical problems.

In [ ]:
class SimpleReActAgent:
    """Minimal ReAct agent for comparison (no planning phase)."""
    
    def __init__(self, max_iterations: int = 8):
        self.max_iterations = max_iterations
        self.total_iterations = 0
    
    def run(self, task: str) -> str:
        system_msg = f"""You are a ReAct agent. Solve the task step by step.

Available tools:
{TOOL_DESC}

Format:
Thought: what you're thinking
Action: tool_name
Input: tool_input

After getting an Observation, continue with Thought/Action or give:
Final Answer: your answer"""
        
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": task}
        ]
        
        for i in range(self.max_iterations):
            self.total_iterations += 1
            response = generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)
            
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip()
            
            action_match = re.search(r'Action:\s*(.+)', response)
            input_match = re.search(r'Input:\s*(.+)', response)
            
            if action_match and input_match:
                tool_name = action_match.group(1).strip()
                tool_input = input_match.group(1).strip()
                if tool_name in TOOLS:
                    try:
                        obs = TOOLS[tool_name]["function"](tool_input)
                    except Exception as e:
                        obs = f"Error: {e}"
                else:
                    obs = f"Unknown tool: {tool_name}"
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {obs}"})
            else:
                return response.strip()
        
        return "Max iterations reached."

print("✓ SimpleReActAgent defined")

In [ ]:
# Run comparison on 3 tasks
comparison_tasks = [
    "What is the total driving distance from NYC to LA to Chicago back to NYC?",
    "Calculate the cost of driving 500 miles if gas costs $3.50/gallon and a car gets 25 mpg.",
    "Compare the speed of light to the average speed of a commercial airplane (550 mph). How many times faster is light?",
]

results_table = []

for i, task in enumerate(comparison_tasks):
    print(f"\n{'='*60}")
    print(f"Task {i+1}: {task}")
    print(f"{'='*60}")
    
    # Plan-and-Execute
    print("\n--- Plan-and-Execute ---")
    pe_agent = PlanAndExecuteAgent()
    pe_start = time.time()
    pe_result = pe_agent.run(task, verbose=False)
    pe_time = time.time() - pe_start
    pe_stats = pe_agent.get_stats()
    print(f"  Result: {pe_result[:150]}...")
    print(f"  Steps: {pe_stats['total_steps']}, Time: {pe_time:.1f}s")
    
    # ReAct
    print("\n--- ReAct ---")
    react_agent = SimpleReActAgent()
    react_start = time.time()
    react_result = react_agent.run(task)
    react_time = time.time() - react_start
    print(f"  Result: {react_result[:150]}...")
    print(f"  Iterations: {react_agent.total_iterations}, Time: {react_time:.1f}s")
    
    results_table.append({
        "task": task[:50] + "...",
        "pe_steps": pe_stats["total_steps"],
        "pe_time": f"{pe_time:.1f}s",
        "react_iters": react_agent.total_iterations,
        "react_time": f"{react_time:.1f}s",
    })

# Print comparison table
print(f"\n\n{'='*80}")
print("COMPARISON TABLE: Plan-and-Execute vs ReAct")
print(f"{'='*80}")
print(f"{'Task':<52} {'P&E Steps':>10} {'P&E Time':>10} {'ReAct Iter':>11} {'ReAct Time':>11}")
print("-" * 94)
for r in results_table:
    print(f"{r['task']:<52} {r['pe_steps']:>10} {r['pe_time']:>10} {r['react_iters']:>11} {r['react_time']:>11}")
print("-" * 94)

## 8. When to Use Plan-and-Execute

### Decision Guide

| Factor | Favor P&E | Favor ReAct |
|--------|-----------|-------------|
| Task complexity | High (5+ steps) | Low (1-3 steps) |
| Step dependencies | Many cross-step deps | Steps are independent |
| Error cost | High (expensive actions) | Low (easy to retry) |
| Need for transparency | High (show plan to user) | Low |
| Adaptability needed | Low-medium | High (rapidly changing) |
| Token budget | Larger (planning overhead) | Smaller |

### Key Trade-offs

1. **Planning overhead**: P&E makes extra LLM calls for planning. For simple tasks, this overhead isn't worth it.
2. **Plan rigidity**: A plan can become stale if the environment changes. Adaptive re-planning mitigates this.
3. **Plan quality**: The planner may create a bad plan. The system is only as good as the planning step.
4. **Parallelism potential**: With a dependency graph, independent steps could be executed in parallel.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## 9. Key Takeaways

1. **Plan-and-Execute separates strategy from tactics** — the Planner thinks about *what* to do; the Executor figures out *how* to do it.

2. **Planning as structured output** — the Planner generates JSON with steps and dependencies, making plans machine-readable and inspectable.

3. **Dependency awareness** — unlike ReAct, P&E understands that step 3 needs results from steps 1 and 2, preventing wasted work.

4. **Adaptive re-planning** — when steps fail, the system can create a new plan that incorporates completed work and avoids the failure mode.

5. **P&E shines on complex, multi-step tasks** — for simple queries, ReAct's lower overhead wins. For tasks with 5+ dependent steps, P&E's global view is valuable.

6. **The pattern generalizes** — any decompose → execute → synthesize workflow is a form of Plan-and-Execute. This applies to code generation, research, data analysis, and more.

---

**Next notebook**: We'll explore how agents can **improve their own output** through reflection and self-critique (Notebook 07).

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory